# 01 — Fit a sphere (Solution)

**Time:** 65 minutes
**Question:** what can a SANS curve tell us about a dilute spherical particle?

You will connect the analytical sphere amplitude to a SasModels fit, choose
bounds, and diagnose the result using normalized residuals.

> **Lecture connection**
>
> - Slide 20, *The SANS master equation*: the model you fit is this equation
>   with $S(Q)=1$ (dilute particles).
> - Slide 23, *The form factor, P(Q)*: the phase-map picture of why a sphere
>   gives minima at $QR \approx 4.49, 7.73, \ldots$ Rewatch the
>   [phase-map video](https://www.youtube.com/watch?v=4VKR-0b25Og) if the formula below feels abstract.
> - Slides 30 and 32, *Step 0: understand your sample* and *Fitting strategy*:
>   where defensible starting values and bounds come from.
> - Slide 33, *Is my fit good or not?*: what this notebook asks you to judge.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

DATA = ROOT / 'data'
print(f'Using sans_fitter from {Path(sans_fitter.__file__).resolve()}')
%matplotlib inline


In [ ]:
values = np.genfromtxt(DATA / 'sphere_reference.csv', delimiter=',', names=True)
q, intensity, uncertainty = values['Q'], values['I'], values['dI']


## Exercise 1 — Reduce the model

For a homogeneous sphere,

$$
F(q)=3\frac{\sin(qr)-qr\cos(qr)}{(qr)^3}, \qquad
I(q)=\mathrm{scale}\,V(\Delta\rho)^2F(q)^2+\mathrm{background}.
$$

Explain why volume is not an independent fifth shape parameter, then complete
the amplitude function. Ignore absolute scaling in this first visual test.

**How this maps onto the lecture.** SasModels writes the sphere as
$I(q) = \frac{\mathrm{scale}}{V}\,\left[3V\Delta\rho\,\frac{\sin(qr)-qr\cos(qr)}{(qr)^3}\right]^2
+ \mathrm{background}$, which is the expression above. Compare it with the
master equation on slide 20,
$I(Q) = \phi\,V_p\,(\Delta\rho)^2\,P(Q)\,S(Q) + \mathrm{bkg}$, where
$P(Q) = F(Q)^2$ is normalized so that $P(0)=1$. For dilute particles $S(Q)=1$,
and on an absolute scale `scale` *is* the volume fraction $\phi$. Because the
workshop data are in arbitrary units, `scale` here also absorbs a calibration
factor. That is one face of the $\phi \leftrightarrow \Delta\rho \leftrightarrow
\mathrm{scale}$ degeneracy from the lecture's part 2 wrap-up (slide 35). We fix
both SLDs so that `scale` carries the rest.

*Hint for the volume question:* write $V$ in terms of $r$.

In [ ]:
def sphere_amplitude(q, radius):
    # Sphere form-factor amplitude F(q), written exactly as in the equation above.
    qr = q * radius
    return 3 * (np.sin(qr) - qr * np.cos(qr)) / qr**3


# The measured intensity follows the amplitude squared, P(q) = F(q)^2.
# trial_radius is in Å. Change this number, then rerun this cell and the plot
# below, until the minima of the model line up with the minima of the data.
trial_radius = 58.0
trial_shape = sphere_amplitude(q, trial_radius) ** 2


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(q, intensity / intensity.max(), 'o', ms=3, label='normalized data')
ax.loglog(q, trial_shape / trial_shape.max(), label=f'trial r = {trial_radius:g} Å')
ax.set(xlabel=r'$q$ / Å$^{-1}$', ylabel='normalized intensity')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


### Reading the size off the curve

Change `trial_radius` until the first minimum of your trial curve lines up
with the first minimum of the data. Then check yourself with the lecture's
rule of thumb (slide 23): the first zero of $F(qr)$ is at $qr \approx 4.493$,
so $r \approx 4.49/q_1$.

- The lecture's worked example used $R = 80$ Å, with minima at
  $Q = 0.0562, 0.0966, 0.136$ Å⁻¹. Are the data minima to the left or right of
  those? Is this sphere bigger or smaller than 80 Å?
- Why are the data minima not exactly zero? Think about noise, the background,
  and, for real samples, polydispersity and instrumental resolution (slide 23's
  dQ/Q panels and slide 41).

The radius you read off here is a *model-free* first estimate. Use it as the
starting value in the fit below rather than guessing.

## Exercise 2 — The fitting workflow

Every fit follows the same sequence:

1. create `SANSFitter()`;
2. load a reduced dataset with `load_data(path)`;
3. select a SasModels model with `set_model('model_name')`;
4. inspect its parameters with `get_params()`;
5. configure each parameter with `set_param(...)`;
6. call `fit(...)` and inspect the residuals with `plot_results(...)`;
7. summarize the result with `get_fit_report()`.

The first two steps are done for you below. You will then choose the model,
set the parameters and run the fit yourself, using only the commands listed
here. Each task cell says exactly which lines to add.

In [ ]:
fitter = SANSFitter()
fitter.load_data(str(DATA / 'sphere_reference.csv'))


In [ ]:
# SasModels models are chosen by name, written as a lower-case string.
# 'sphere' is a homogeneous sphere. All names are listed at
# https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/models/index.html
model_name = 'sphere'
fitter.set_model(model_name)

# get_params() prints every parameter of the chosen model: its current value,
# its bounds, and whether it will be fitted (the Vary column).
fitter.get_params()


### Changing parameters

The general form is:

```python
fitter.set_param(
    'parameter_name',
    value=initial_guess,
    min=lower_bound,
    max=upper_bound,
    vary=True,
)
```

- `value` is the optimizer's starting point, not the answer.
- `min` and `max` express physically defensible limits.
- `vary=True` lets the optimizer change the parameter.
- `vary=False` fixes independently known quantities such as SLD.

Keep both SLDs fixed. Choose and justify a starting value and bounds for
radius, scale and background, using the table below. There is no single right
answer; defensible numbers are what matter.

**Where do defensible numbers come from?** Slide 33 says it directly: *start
from the data, not a guess*. The lecture's *Step 0* (slide 30)
is to know what you put in the beam. In practice:

| Parameter | Lecture source for a starting value | Sensible bounds reflect… |
|---|---|---|
| `sld`, `sld_solvent` | Composition and density via an SLD calculator (slide 12) | fixed: known independently |
| `radius` | First minimum, $r \approx 4.49/q_1$ (slide 23), or Guinier, $R = \sqrt{5/3}\,R_g$ (slide 21) | synthesis, TEM, or DLS knowledge |
| `scale` | Height of the low-q plateau, $I(0) \propto \phi V \Delta\rho^2$ (slide 20) | concentration and calibration |
| `background` | Level of the flat high-q floor (slide 15) | must be ≥ 0 and below the data |

The SLD values used here are 4.2 for the particle and 1.0 for the solvent, in
units of 10⁻⁶ Å⁻², the "SLD units" of slide 12.

In [ ]:
# Contrast: both SLDs (in 10^-6 Å^-2) are known from the sample composition,
# so they are fixed with vary=False and the optimizer will not change them.
fitter.set_param('sld', value=4.2, vary=False)
fitter.set_param('sld_solvent', value=1.0, vary=False)

# Fitted parameters: value = starting guess, min/max = physically sensible limits,
# vary=True = let the optimizer adjust it.
# Radius in Å: start near the estimate from the first minimum.
fitter.set_param('radius', value=50, min=20, max=100, vary=True)
# Scale: the overall intensity level (volume fraction x calibration here).
fitter.set_param('scale', value=0.05, min=0.001, max=0.2, vary=True)
# Background: the flat level at high q. It cannot be negative.
fitter.set_param('background', value=0.002, min=0, max=0.01, vary=True)

fitter.get_params()


### Running the optimizer

`fit(engine='lmfit', method='least_squares')` performs a bounded local
least-squares fit. `plot_results(...)` overlays the model and displays
normalized residuals. Afterwards, `get_fit_report()` summarizes the fit in one
table (used in Exercise 3).

**What the optimizer actually does.** It searches for the parameters $\theta$
that minimize

$$
\chi^2(\theta) = \sum_i \left(\frac{I_i - I_{\mathrm{model}}(q_i;\theta)}{\mathrm{d}I_i}\right)^2 .
$$

Each term in brackets is a **normalized residual**: the misfit at one point,
measured in units of its own uncertainty. That is why `dI` matters (notebook
00). As slide 33 puts it, good residuals scatter randomly around 0, mostly
within ±2, and change sign roughly every other point (about $N/2$ sign changes
for $N$ points). Structure in the residuals is the diagnostic, more than the χ²
number ([SasView: *Assessing fit quality*](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/residuals_help.html)).

Two practical notes:

- The engine label `'lmfit'` is historical. Underneath, the helper calls
  [SciPy `least_squares`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html), a bounded trust-region method that only searches downhill from the
  starting point.
- While it runs, `fit()` prints a χ²: the **sum** above for `engine='lmfit'`,
  but the **reduced** value $\chi^2/(N-p)$ for the BUMPS engine ($N$ data
  points, $p$ fitted parameters). The fit report always gives both, calculated
  the same way, so use it to compare fits.

In [ ]:
# Run a bounded least-squares fit, starting from the values set above.
fitter.fit(engine='lmfit', method='least_squares')

# Top panel: data and model. Bottom panel: normalized residuals,
# (data - model) / dI, i.e. the misfit in units of the error bar.
fitter.plot_results(show_residuals=True, log_scale=True)


## Exercise 3 — For your now-expert critique!

Record:

- fitted radius and uncertainty;
- whether residuals look randomly distributed around zero;
- any fitted parameter touching a bound;
- one experimental systematic that this model ignores.

**Checklist for "is my fit good?" (slide 33).**

1. *Residuals:* scattered around zero with no runs or waves? A wave that
   follows the form-factor oscillations means the shape or size distribution
   is wrong, not the noise.
2. *Statistic:* is the reduced χ², $\chi^2_\nu = \chi^2/\nu$ with
   $\nu = N-p$, within about $1 \pm \sqrt{2/\nu}$ (slide 33)? Much larger
   means a missing ingredient or underestimated `dI`. Much smaller means
   overestimated `dI` or overfitting. A χ²ν of 1 with a long run of positive
   residuals is still a bad fit: χ² mixes the model and the error bars.
3. *Bounds:* a parameter pinned at a bound has not been determined. The
   bound has decided it.
4. *Plausibility:* does the radius agree with your model-free estimate from
   the first minimum? Is the background below the lowest data points?

**Systematics a single monodisperse sphere ignores:** a size distribution
([SasView polydispersity](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/pd/polydispersity.html)) and instrumental smearing ([SasView resolution](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/resolution.html)). Both fill in the minima (slide 23 shows 0, 5 and 10 % dQ/Q). The
lecture also covered reduction errors in transmission, empty-cell subtraction
and thickness (slides 44 and 46), and interparticle interactions, $S(Q) \neq 1$
(notebook 03). These synthetic data were generated without any of them. That is
why the minima are so sharp. Say which one you would expect to matter most for a
real measurement, and why.

In [ ]:
# The fit report: every parameter with its uncertainty, whether it was fitted
# or fixed and whether it ended on a bound, plus the slide-33 checks: reduced χ²
# with its expected range, residuals within ±2, and residual sign changes.
report = fitter.get_fit_report()
report

# Other ways to use the same report:
#   print(report)                          plain-text table
#   report.to_markdown()                   Markdown, e.g. for your notes
#   report.parameters['radius']['value']   one number


**Model answer.** The fitted radius is about 58.0 ± 0.02 Å, matching the first-minimum
estimate $4.49/q_1$. The residuals scatter randomly around zero with no wave
following the oscillations: 97 % lie within ±2 and they change sign 49 times,
close to the $N/2 \approx 45$ expected for random scatter. The reduced χ² of
0.76 is slightly below the expected 1 ± 0.15, so the error bars are, if
anything, a little generous. No parameter sits on a bound. The model ignores polydispersity and instrumental resolution. For a real
sample these would fill in the minima, and the quoted ± 0.02 Å would be far too
optimistic.

## Extension — Bad starting values

What happens if the starting radius is poor, say 15 Å, with broad bounds? Does
the local optimizer reach the same answer? Compare it with BUMPS differential
evolution, `fitter.fit(engine='bumps', method='de', steps=200)`. What does this
reveal about optimization versus inference?

*Why this happens:* as slide 33 warns, a fit started far from the answer can be
trapped in a local minimum. The χ² surface of a form factor is full of them.
A sphere of 15 Å puts its first minimum near $4.49/15 \approx 0.3$ Å⁻¹, so
a downhill search starting there can lock onto the wrong oscillation.
[Differential evolution](https://bumps.readthedocs.io/en/latest/guide/optimizer.html#differential-evolution) keeps a population of candidates spread over
the whole prior box, so it is far less sensitive to the start. Neither method
tells you *how well* the radius is determined. That question, the theme of slide
34 (*Deeper into uncertainty*), is taken up in notebook 04.

Remember the χ² note above when you compare the two engines' numbers.

The cell below runs both experiments for you. Compare the two residual plots
and the two printed reports. Which parameter does the first report flag?

In [ ]:
# 1. The same local optimizer, from a deliberately poor start of 15 Å.
fitter.set_param('radius', value=15, min=10, max=200, vary=True)
fitter.fit(engine='lmfit', method='least_squares')
fitter.plot_results(show_residuals=True, log_scale=True)
local_report = fitter.get_fit_report()

# 2. Differential evolution, a population search over the whole min-max range.
fitter.fit(engine='bumps', method='de', steps=200)
fitter.plot_results(show_residuals=True, log_scale=True)
global_report = fitter.get_fit_report()

print('LOCAL OPTIMIZER FROM 15 Å')
print(local_report)
print()
print('DIFFERENTIAL EVOLUTION')
print(global_report)


## Further reading

- [Gommes, Jaksch & Frielinghaus, *Small-angle scattering for beginners*, J. Appl. Cryst. 54, 1832 (2021)](https://pmc.ncbi.nlm.nih.gov/articles/PMC8662971/), sections on the form factor and the sphere.
- [SasView `sphere` model documentation](https://www.sasview.org/docs/user/models/sphere.html): the exact
  equation, parameter units, and the meaning of `scale` on an absolute scale.
- J. S. Pedersen, *Analysis of small-angle scattering data from colloids and polymer
  solutions: modeling and least-squares fitting*, Adv. Colloid Interface Sci. 70, 171
  (1997): the standard reference for form factors, polydispersity and smearing in fits.